# RQ4: How does encoder performance vary across different target datasets?

## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'clean_saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    # technique_summary = technique_summary.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    technique_summary = technique_summary.sort_values(
        ["Mean","Net Score"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    # backbone_totals = backbone_totals.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    backbone_totals = backbone_totals.sort_values(
        [ "Mean","Net Score"], 
        ascending=[False, False]
    )
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



In [ ]:
summarized_executions_path

In [ ]:
df = pd.read_csv(summarized_executions_path)
df

In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Freeze')]

df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('KH')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('ResNet-SE-5')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")


In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('KH')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('ResNet-SE-5')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")


In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

# df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('WISDM')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('CNN-PFF')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")

In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('WISDM')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('CNN-PFF')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")

In [ ]:
rows = []

for technique in sorted(df['tsk_pretext'].unique()):
    df_t = df[df['tsk_pretext'] == technique]
    df_t = df_t[df_t['ft_strategy'].str.contains('Full Finetune')]

    for backbone in ['ResNet-SE-5','CNN-PFF','TS2Vec Encoder','TS-TCC Encoder','IMU Transformer','RNN']:
    # for backbone in sorted(df_t['backbone'].unique()):
        df_b = df_t[df_t['backbone'] == backbone]

        for frac in sorted(df_b['d_pretext'].unique()):
            df_f = df_b[df_b['d_pretext'] == frac]

            mean_acc = df_f['metric'].mean() * 100
            std_acc  = df_f['metric'].std() * 100
            n_runs   = df_f['metric'].count()

            rows.append({
                'tsk_pretext': technique,
                'backbone': backbone,
                'd_pretext': frac,
                'mean_acc': mean_acc,
                'std_acc': std_acc,
                'acc': f"{mean_acc:.1f} ± {std_acc:.1f}",
                'n': n_runs
            })
summary_loop = pd.DataFrame(rows)
display(summary_loop)


In [ ]:
TECHNIQUE_ORDER = [
    'TNC',
    'TFC',
    'Diet',
    'LFR',
    'Supervised',
]

FRAC_ORDER = ['UCI','RW-Thigh','RW-Waist','MS','WISDM','KH']

BACKBONE_ORDER = [
    'ResNet-SE-5',
    'CNN-PFF',
    'TS2Vec Encoder',
    'TS-TCC Encoder',
    'IMU Transformer',
    'RNN',
]

summary_loop['tsk_pretext'] = pd.Categorical(
    summary_loop['tsk_pretext'],
    categories=TECHNIQUE_ORDER,
    ordered=True
)

summary_loop['d_pretext'] = pd.Categorical(
    summary_loop['d_pretext'],
    categories=FRAC_ORDER,
    ordered=True
)

summary_loop['backbone'] = pd.Categorical(
    summary_loop['backbone'],
    categories=BACKBONE_ORDER,
    ordered=True
)


In [ ]:
sanity_table_loop = summary_loop.pivot_table(
    index=['tsk_pretext', 'd_pretext'],
    columns='backbone',
    values='acc',
    aggfunc='first'
).sort_index()

display(sanity_table_loop)


In [ ]:
rows = []

for technique in TECHNIQUE_ORDER:
    df_t = df[
        (df['tsk_pretext'] == technique) &
        (df['ft_strategy'].str.contains('Full Finetune', na=False))
    ]

    for backbone in BACKBONE_ORDER:
        df_tb = df_t[df_t['backbone'] == backbone]

        if df_tb.empty:
            continue

        mean_acc = df_tb['metric'].mean() * 100
        std_acc  = df_tb['metric'].std() * 100
        n_runs   = df_tb['metric'].count()

        rows.append({
            'tsk_pretext': technique,
            'backbone': backbone,
            'mean_acc': mean_acc,
            'std_acc': std_acc,
            'acc': f"{mean_acc:.1f} ± {std_acc:.1f}",
            'n': n_runs
        })

summary_encoder_avg = pd.DataFrame(rows)
display(summary_encoder_avg)


In [ ]:
# FREEZE

rows = []

for technique in sorted(df['tsk_pretext'].unique()):
    df_t = df[df['tsk_pretext'] == technique]
    df_t = df_t[df_t['ft_strategy'].str.contains('Freeze')]

    for backbone in ['ResNet-SE-5','CNN-PFF','TS2Vec Encoder','TS-TCC Encoder','IMU Transformer','RNN']:
    # for backbone in sorted(df_t['backbone'].unique()):
        df_b = df_t[df_t['backbone'] == backbone]

        for frac in sorted(df_b['d_pretext'].unique()):
            df_f = df_b[df_b['d_pretext'] == frac]

            mean_acc = df_f['metric'].mean() * 100
            std_acc  = df_f['metric'].std() * 100
            n_runs   = df_f['metric'].count()

            rows.append({
                'tsk_pretext': technique,
                'backbone': backbone,
                'd_pretext': frac,
                'mean_acc': mean_acc,
                'std_acc': std_acc,
                'acc': f"{mean_acc:.1f} ± {std_acc:.1f}",
                'n': n_runs
            })
summary_loop = pd.DataFrame(rows)
display(summary_loop)
TECHNIQUE_ORDER = [
    'TNC',
    'TFC',
    'Diet',
    'LFR',
    'Supervised',
]

FRAC_ORDER = ['UCI','RW-Thigh','RW-Waist','MS','WISDM','KH']

BACKBONE_ORDER = [
    'ResNet-SE-5',
    'CNN-PFF',
    'TS2Vec Encoder',
    'TS-TCC Encoder',
    'IMU Transformer',
    'RNN',
]

summary_loop['tsk_pretext'] = pd.Categorical(
    summary_loop['tsk_pretext'],
    categories=TECHNIQUE_ORDER,
    ordered=True
)

summary_loop['d_pretext'] = pd.Categorical(
    summary_loop['d_pretext'],
    categories=FRAC_ORDER,
    ordered=True
)

summary_loop['backbone'] = pd.Categorical(
    summary_loop['backbone'],
    categories=BACKBONE_ORDER,
    ordered=True
)

sanity_table_loop = summary_loop.pivot_table(
    index=['tsk_pretext', 'd_pretext'],
    columns='backbone',
    values='acc',
    aggfunc='first'
).sort_index()

display(sanity_table_loop)



In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Freeze')]

# df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('WISDM')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('CNN-PFF')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")

In [ ]:
rows = []

for technique in TECHNIQUE_ORDER:
    df_t = df[
        (df['tsk_pretext'] == technique) &
        (df['ft_strategy'].str.contains('Freeze', na=False))
    ]

    for backbone in BACKBONE_ORDER:
        df_tb = df_t[df_t['backbone'] == backbone]

        if df_tb.empty:
            continue

        mean_acc = df_tb['metric'].mean() * 100
        std_acc  = df_tb['metric'].std() * 100
        n_runs   = df_tb['metric'].count()

        rows.append({
            'tsk_pretext': technique,
            'backbone': backbone,
            'mean_acc': mean_acc,
            'std_acc': std_acc,
            'acc': f"{mean_acc:.1f} ± {std_acc:.1f}",
            'n': n_runs
        })

summary_encoder_avg = pd.DataFrame(rows)
display(summary_encoder_avg)

# display only acc and tsk pretext
display(summary_encoder_avg[['tsk_pretext','backbone','acc']])


backbones cnn better but a lot of variance we need to take a close look on whats happening

### P4 Qual o melhor backbone para HAR de acordo com o dataset?

Qual o melhor backbone para o TNC em cada dataset?
Qual o melhor backbone para o TFC em cada dataset?
Qual o melhor backbone para o Diet em cada dataset?
Qual o melhor backbone para o LFR em cada dataset?
Qual o melhor backbone para SL em cada dataset?


In [ ]:
df['d_target'].unique()

In [ ]:
base_df = df.copy()

In [ ]:
base_df["frac_dtarget"].unique()

In [ ]:
base_df['ft_strategy'].unique()
import numpy as np

In [ ]:
# freeze by dataset
df = base_df
datasets = ['UCI']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TNC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    display(df_variant)
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]

technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



In [ ]:
# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Freeze"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]

technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)



In [ ]:
# with ts2vec 

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            # "select_tsk_pretext": ["TNC","Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
df_plot_ = combined_df_rq4_by_dataset.copy()
# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
df_plot = df_plot_.copy()

In [ ]:
df_plot_

In [ ]:
df_plot["Mean"] *= 100  # convert to percentage
df_plot["Std"] *= 100   # convert to percentage
# Split Backbone column into Backbone and Dataset for better plotting
df_plot[["Backbone_only", "Dataset_only"]] = df_plot["Backbone"].str.split(" \+ ", expand=True)

df_plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Custom dataset order
dataset_order = ["KH", "MS", "RW-Thigh", "RW-Waist", "UCI", "WISDM"]

# Define order and palette for backbones
# backbone_order = ["RNN", "IMU Transformer", "ResNet-1D", "CNN-PFF", "TS Encoder"]
# backbone_order = ["RNN", "IMU Transformer", "ResNet-SE-5" ,"CNN-PFF",'TS2Vec Encoder','TS-TCC Encoder']
backbone_order = ["ResNet-SE-5" ,'TS-TCC Encoder',"CNN-PFF",'TS2Vec Encoder', "IMU Transformer", "RNN"]
palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:red",
    # "ResNet-1D": "tab:green",
    "ResNet-SE-5": "tab:green",
    "CNN-PFF": "tab:orange",
    "TS2Vec Encoder": "tab:purple",
    'TS-TCC Encoder': 'tab:brown',
}

plt.figure(figsize=(12, 6))
sns.set(style="whitegrid", font_scale=1.2)

# Calculate positions
n_datasets = len(dataset_order)
n_backbones = len(backbone_order)
bar_width = 0.12
spacing = 0
x_pos = np.arange(n_datasets)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 6))

for i, backbone in enumerate(backbone_order):
    means = []
    stds = []
    for dataset in dataset_order:
        # Find the data for this backbone + dataset combination
        mask = (df_plot['Dataset_only'] == dataset) & (df_plot['Backbone_only'] == backbone)
        if mask.any():
            row = df_plot[mask].iloc[0]
            means.append(row['Mean'])
            stds.append(row['Std'])
        else:
            means.append(0)
            stds.append(0)
    
    # Calculate bar positions
    positions = x_pos + i * (bar_width + spacing)
    
    # Plot bars with error bars
    bars = ax.bar(positions, means, bar_width, label=backbone, 
                 color=palette[backbone], yerr=stds, capsize=4, 
                 error_kw={'elinewidth': 2, 'capthick': 2})

# Customize the plot
ax.set_ylabel("Accuracy (%)", fontsize=14, fontweight='bold')
ax.set_xlabel("Dataset", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos + (n_backbones - 1) * (bar_width + spacing) / 2)
ax.set_xticklabels(dataset_order, rotation=30, fontsize=12)
ax.set_ylim(0, 100)

# Position legend on the right side, centered vertically
ax.legend(title="Encoder", fontsize=12, title_fontsize=12, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.savefig("backbone_dataset_performance.png", dpi=300, bbox_inches='tight')
plt.show()

### freeze

In [ ]:
# with ts2vec ssl freeze

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Freeze"],
            "select_tsk_pretext": ["TNC","Diet","TFC","LFR"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
display(combined_df_rq4_by_dataset)
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

## split by technique

In [ ]:
# Create a comprehensive table with Full Finetune top and Freeze bottom
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
encoders = ['ResNet-SE-5', 'CNN-PFF', 'TS2Vec Encoder', 'TS-TCC Encoder', 'IMU Transformer', 'RNN']
pretext_tasks = ['TNC', 'TFC', 'Diet', 'LFR', 'Supervised']

# Collect all results
all_results = []

for dataset in datasets:
    for pretext_task in pretext_tasks:
        for ft_strategy in ['Freeze', 'Full Finetune']:
            try:
                dot, df_variant = show_precedence_graph(
                    df,
                    variants_variables=["backbone", "d_target"],
                    filters={
                        "select_d_target": [dataset],
                        "select_tsk_pretext": [pretext_task],
                        "select_ft_strategy": [ft_strategy],
                    },
                    show_stdev=True,
                    apply_bonferroni=apply_correction_factor,
                )
                
                if not df_variant.empty:
                    summary_df = summarize_backbone_performance(df_variant)
                    summary_df["Dataset"] = dataset
                    summary_df["Pretext Task"] = pretext_task
                    summary_df["FT Strategy"] = ft_strategy
                    all_results.append(summary_df)
                    display(summary_df)
            except Exception as e:
                print(f"Error processing {dataset}, {pretext_task}, {ft_strategy}: {e}")
                continue

# Combine all results
combined_results = pd.concat(all_results, ignore_index=True)

# Clean the backbone names properly
def clean_backbone_name(backbone_str):
    """Extract clean backbone name from the format 'Backbone + Dataset'"""
    if ' + ' in backbone_str:
        return backbone_str.split(' + ')[0].strip()
    return backbone_str.strip()

combined_results['Backbone Clean'] = combined_results['Backbone'].apply(clean_backbone_name)

# Create the comprehensive LaTeX table
print("=== Comprehensive LaTeX Table: Full Finetune (Top) and Freeze (Bottom) ===")
print()

print("\\begin{table}[!htbp]")
print("\\centering")
print("\\scriptsize")
print("\\setlength{\\tabcolsep}{3pt}")
print("\\begin{tabular}{@{}lcccccc@{}}")
print("\\toprule")
print("\\textbf{SSL Technique} & \\multicolumn{6}{c}{\\textbf{Encoders}} \\\\")
print("\\cmidrule(lr){2-7}")
print("& \\textbf{ResNet-SE-5} & \\textbf{CNN-PFF} & \\textbf{TS2Vec} & \\textbf{TS-TCC} & \\textbf{IMU Trans.} & \\textbf{RNN} \\\\")
print("\\midrule")

print("\\multicolumn{7}{l}{\\textbf{Full Finetune}} \\\\")
print("\\cmidrule(lr){1-7}")

# FULL FINETUNE SECTION
for ssl_tech in pretext_tasks:
    finetune_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Full Finetune')
    ]
    
    ssl_row = f"{ssl_tech:<12}"
    for encoder in encoders:
        encoder_data = finetune_data[finetune_data['Backbone Clean'] == encoder]
        if not encoder_data.empty:
            # Calculate average across all datasets for this encoder and SSL technique
            mean_avg = encoder_data['Mean'].mean() * 100
            std_avg = encoder_data['Std'].mean() * 100
            net_score_total = encoder_data['Net Score'].sum()
            ssl_row += f"& {mean_avg:.1f}±{std_avg:.1f} ({net_score_total:+d}) "
        else:
            ssl_row += "& - "
    ssl_row += "\\\\"
    print(ssl_row)

# Full Finetune Average across all SSL techniques
print("\\cmidrule(lr){1-7}")
finetune_avg_row = "\\textbf{Average}"
for encoder in encoders:
    encoder_all_finetune = combined_results[
        (combined_results['Backbone Clean'] == encoder) & 
        (combined_results['FT Strategy'] == 'Full Finetune')
    ]
    if not encoder_all_finetune.empty:
        mean_avg = encoder_all_finetune['Mean'].mean() * 100
        std_avg = encoder_all_finetune['Std'].mean() * 100
        net_score_total = encoder_all_finetune['Net Score'].sum()
        finetune_avg_row += f"& \\textbf{{{mean_avg:.1f}±{std_avg:.1f}}} (\\textbf{{{net_score_total:+d}}}) "
    else:
        finetune_avg_row += "& - "
finetune_avg_row += "\\\\"
print(finetune_avg_row)

print("\\midrule")
print("\\multicolumn{7}{l}{\\textbf{Freeze}} \\\\")
print("\\cmidrule(lr){1-7}")

# FREEZE SECTION
for ssl_tech in pretext_tasks:
    freeze_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Freeze')
    ]
    
    ssl_row = f"{ssl_tech:<12}"
    for encoder in encoders:
        encoder_data = freeze_data[freeze_data['Backbone Clean'] == encoder]
        if not encoder_data.empty:
            # Calculate average across all datasets for this encoder and SSL technique
            mean_avg = encoder_data['Mean'].mean() * 100
            std_avg = encoder_data['Std'].mean() * 100
            net_score_total = encoder_data['Net Score'].sum()
            ssl_row += f"& {mean_avg:.1f}±{std_avg:.1f} ({net_score_total:+d}) "
        else:
            ssl_row += "& - "
    ssl_row += "\\\\"
    print(ssl_row)

# Freeze Average across all SSL techniques
print("\\cmidrule(lr){1-7}")
freeze_avg_row = "\\textbf{Average}"
for encoder in encoders:
    encoder_all_freeze = combined_results[
        (combined_results['Backbone Clean'] == encoder) & 
        (combined_results['FT Strategy'] == 'Freeze')
    ]
    if not encoder_all_freeze.empty:
        mean_avg = encoder_all_freeze['Mean'].mean() * 100
        std_avg = encoder_all_freeze['Std'].mean() * 100
        net_score_total = encoder_all_freeze['Net Score'].sum()
        freeze_avg_row += f"& \\textbf{{{mean_avg:.1f}±{std_avg:.1f}}} (\\textbf{{{net_score_total:+d}}}) "
    else:
        freeze_avg_row += "& - "
freeze_avg_row += "\\\\"
print(freeze_avg_row)

print("\\bottomrule")
print("\\end{tabular}")
print("\\caption{Encoder performance across SSL techniques and refinement strategies. Top: Full Finetune results. Bottom: Freeze results. Mean±Std accuracy (\\%) averaged across all datasets and net score (summed across datasets) in parentheses. Bold values show averages across all SSL techniques.}")
print("\\label{tab:finetune_freeze_comprehensive}")
print("\\end{table}")

# Alternative: More detailed version with per-dataset information
print("\n" + "="*80)
print("=== Detailed Version with Per-Dataset Breakdown ===")
print()

print("\\begin{table}[!htbp]")
print("\\centering")
print("\\tiny")
print("\\setlength{\\tabcolsep}{1.5pt}")
print("\\begin{tabular}{@{}llcccccc@{}}")
print("\\toprule")
print("\\textbf{Strategy} & \\textbf{SSL/Dataset} & \\multicolumn{6}{c}{\\textbf{Encoders}} \\\\")
print("\\cmidrule(lr){3-8}")
print("& & \\textbf{ResNet} & \\textbf{CNN-PFF} & \\textbf{TS2Vec} & \\textbf{TS-TCC} & \\textbf{IMU Trans.} & \\textbf{RNN} \\\\")
print("\\midrule")

# FULL FINETUNE SECTION
print("\\multirow{31}{*}{\\rotatebox{90}{\\textbf{Full Finetune}}}")

for ssl_tech in pretext_tasks:
    # SSL technique average row
    finetune_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Full Finetune')
    ]
    
    ssl_avg_row = f"& \\textbf{{{ssl_tech}}}"
    for encoder in encoders:
        encoder_data = finetune_data[finetune_data['Backbone Clean'] == encoder]
        if not encoder_data.empty:
            mean_avg = encoder_data['Mean'].mean() * 100
            std_avg = encoder_data['Std'].mean() * 100
            net_score_total = encoder_data['Net Score'].sum()
            ssl_avg_row += f"& \\textbf{{{mean_avg:.1f}±{std_avg:.1f}}} (\\textbf{{{net_score_total:+d}}}) "
        else:
            ssl_avg_row += "& - "
    ssl_avg_row += "\\\\"
    print(ssl_avg_row)
    
    # Individual datasets for this SSL technique
    for dataset in datasets:
        dataset_data = combined_results[
            (combined_results['Pretext Task'] == ssl_tech) & 
            (combined_results['FT Strategy'] == 'Full Finetune') &
            (combined_results['Dataset'] == dataset)
        ]
        
        if not dataset_data.empty:
            dataset_row = f"& \\quad {dataset}"
            for encoder in encoders:
                encoder_data = dataset_data[dataset_data['Backbone Clean'] == encoder]
                if not encoder_data.empty:
                    row = encoder_data.iloc[0]
                    mean_pct = row['Mean'] * 100
                    std_pct = row['Std'] * 100
                    net_score = row['Net Score']
                    dataset_row += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
                else:
                    dataset_row += "& - "
            dataset_row += "\\\\"
            print(dataset_row)
    
    if ssl_tech != pretext_tasks[-1]:
        print("\\cmidrule(lr){2-8}")

print("\\midrule")

# FREEZE SECTION
print("\\multirow{31}{*}{\\rotatebox{90}{\\textbf{Freeze}}}")

for ssl_tech in pretext_tasks:
    # SSL technique average row
    freeze_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Freeze')
    ]
    
    ssl_avg_row = f"& \\textbf{{{ssl_tech}}}"
    for encoder in encoders:
        encoder_data = freeze_data[freeze_data['Backbone Clean'] == encoder]
        if not encoder_data.empty:
            mean_avg = encoder_data['Mean'].mean() * 100
            std_avg = encoder_data['Std'].mean() * 100
            net_score_total = encoder_data['Net Score'].sum()
            ssl_avg_row += f"& \\textbf{{{mean_avg:.1f}±{std_avg:.1f}}} (\\textbf{{{net_score_total:+d}}}) "
        else:
            ssl_avg_row += "& - "
    ssl_avg_row += "\\\\"
    print(ssl_avg_row)
    
    # Individual datasets for this SSL technique
    for dataset in datasets:
        dataset_data = combined_results[
            (combined_results['Pretext Task'] == ssl_tech) & 
            (combined_results['FT Strategy'] == 'Freeze') &
            (combined_results['Dataset'] == dataset)
        ]
        
        if not dataset_data.empty:
            dataset_row = f"& \\quad {dataset}"
            for encoder in encoders:
                encoder_data = dataset_data[dataset_data['Backbone Clean'] == encoder]
                if not encoder_data.empty:
                    row = encoder_data.iloc[0]
                    mean_pct = row['Mean'] * 100
                    std_pct = row['Std'] * 100
                    net_score = row['Net Score']
                    dataset_row += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
                else:
                    dataset_row += "& - "
            dataset_row += "\\\\"
            print(dataset_row)
    
    if ssl_tech != pretext_tasks[-1]:
        print("\\cmidrule(lr){2-8}")

print("\\bottomrule")
print("\\end{tabular}")
print("\\caption{Detailed encoder performance with Full Finetune (top) and Freeze (bottom) strategies. For each SSL technique, bold rows show averages across all datasets, while individual dataset results are shown indented. Mean±Std accuracy (\\%) and net score in parentheses.}")
print("\\label{tab:detailed_finetune_freeze}")
print("\\end{table}")

In [ ]:
# Create a comprehensive table split by SSL technique, encoder, and dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
encoders = ['ResNet-SE-5', 'CNN-PFF', 'TS2Vec Encoder', 'TS-TCC Encoder', 'IMU Transformer', 'RNN']
pretext_tasks = ['TNC', 'TFC', 'Diet', 'LFR', 'Supervised']

# Collect all results
all_results = []

for dataset in datasets:
    for pretext_task in pretext_tasks:
        for ft_strategy in ['Freeze', 'Full Finetune']:
            try:
                dot, df_variant = show_precedence_graph(
                    df,
                    variants_variables=["backbone", "d_target"],
                    filters={
                        "select_d_target": [dataset],
                        "select_tsk_pretext": [pretext_task],
                        "select_ft_strategy": [ft_strategy],
                    },
                    show_stdev=True,
                    apply_bonferroni=apply_correction_factor,
                )
                
                if not df_variant.empty:
                    summary_df = summarize_backbone_performance(df_variant)
                    summary_df["Dataset"] = dataset
                    summary_df["Pretext Task"] = pretext_task
                    summary_df["FT Strategy"] = ft_strategy
                    all_results.append(summary_df)
            except Exception as e:
                print(f"Error processing {dataset}, {pretext_task}, {ft_strategy}: {e}")
                continue

# Combine all results
combined_results = pd.concat(all_results, ignore_index=True)

# Clean the backbone names properly
def clean_backbone_name(backbone_str):
    """Extract clean backbone name from the format 'Backbone + Dataset'"""
    if ' + ' in backbone_str:
        return backbone_str.split(' + ')[0].strip()
    return backbone_str.strip()

combined_results['Backbone Clean'] = combined_results['Backbone'].apply(clean_backbone_name)

# Debug: Check what backbones we have
print("Available backbones in data:")
print(combined_results['Backbone Clean'].unique())
print("\nAvailable datasets:")
print(combined_results['Dataset'].unique())
print("\nAvailable pretext tasks:")
print(combined_results['Pretext Task'].unique())

# Create LaTeX table - Full Finetune First
print("=== LaTeX Table: SSL Technique × Encoder × Dataset (Full Finetune First) ===")
print()

# Start LaTeX table
print("\\begin{table}[!htbp]")
print("\\centering")
print("\\scriptsize")
print("\\setlength{\\tabcolsep}{3pt}")
print("\\begin{tabular}{@{}lcccccc@{}}")
print("\\toprule")
print("\\textbf{SSL Technique} & \\multicolumn{6}{c}{\\textbf{Encoders}} \\\\")
print("\\cmidrule(lr){2-7}")
print("& \\textbf{ResNet-SE-5} & \\textbf{CNN-PFF} & \\textbf{TS2Vec} & \\textbf{TS-TCC} & \\textbf{IMU Trans.} & \\textbf{RNN} \\\\")
print("\\midrule")

# Group by SSL technique
for ssl_tech in pretext_tasks:
    print(f"\\textbf{{{ssl_tech}}} \\\\")
    
    # Full Finetune first
    finetune_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Full Finetune')
    ]
    
    if not finetune_data.empty:
        finetune_row = "Full Finetune"
        for encoder in encoders:
            encoder_data = finetune_data[finetune_data['Backbone Clean'] == encoder]
            if not encoder_data.empty:
                # Take the first result (should be the same across datasets for this analysis)
                row = encoder_data.iloc[0]
                mean_pct = row['Mean'] * 100
                std_pct = row['Std'] * 100
                net_score = row['Net Score']
                finetune_row += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
            else:
                finetune_row += "& - "
        finetune_row += "\\\\"
        print(finetune_row)
    
    # Then Freeze
    freeze_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Freeze')
    ]
    
    if not freeze_data.empty:
        freeze_row = "Freeze"
        for encoder in encoders:
            encoder_data = freeze_data[freeze_data['Backbone Clean'] == encoder]
            if not encoder_data.empty:
                row = encoder_data.iloc[0]
                mean_pct = row['Mean'] * 100
                std_pct = row['Std'] * 100
                net_score = row['Net Score']
                freeze_row += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
            else:
                freeze_row += "& - "
        freeze_row += "\\\\"
        print(freeze_row)
    else:
        # If no freeze data, just show dashes
        freeze_row = "Freeze & - & - & - & - & - & - \\\\"
        print(freeze_row)
    
    # Add space between SSL techniques
    if ssl_tech != pretext_tasks[-1]:
        print("\\addlinespace[0.3em]")

print("\\bottomrule")
print("\\end{tabular}")
print("\\caption{Encoder performance across SSL techniques and refinement strategies. Mean±Std accuracy (\\%) and net score in parentheses.}")
print("\\label{tab:ssl_encoder_strategies}")
print("\\end{table}")

# Create a more detailed version showing per dataset
print("\n" + "="*80)
print("=== Detailed LaTeX Table: Per Dataset Performance ===")
print()

for ssl_tech in pretext_tasks:
    print(f"\\begin{{table}}[!htbp]")
    print("\\centering")
    print("\\scriptsize")
    print("\\setlength{\\tabcolsep}{3pt}")
    print("\\begin{tabular}{@{}lcccccc@{}}")
    print("\\toprule")
    print(f"\\textbf{{{ssl_tech}}} & \\multicolumn{{6}}{{c}}{{\\textbf{{Encoders}}}} \\\\")
    print("\\cmidrule(lr){2-7}")
    print("\\textbf{Dataset} & \\textbf{ResNet-SE-5} & \\textbf{CNN-PFF} & \\textbf{TS2Vec} & \\textbf{TS-TCC} & \\textbf{IMU Trans.} & \\textbf{RNN} \\\\")
    print("\\midrule")
    
    # Full Finetune section
    print("\\multicolumn{7}{l}{\\textbf{Full Finetune}} \\\\")
    finetune_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Full Finetune')
    ]
    
    for dataset in datasets:
        dataset_finetune = finetune_data[finetune_data['Dataset'] == dataset]
        if not dataset_finetune.empty:
            row_str = f"{dataset:<10}"
            for encoder in encoders:
                encoder_data = dataset_finetune[dataset_finetune['Backbone Clean'] == encoder]
                if not encoder_data.empty:
                    row = encoder_data.iloc[0]
                    mean_pct = row['Mean'] * 100
                    std_pct = row['Std'] * 100
                    net_score = row['Net Score']
                    row_str += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
                else:
                    row_str += "& - "
            row_str += "\\\\"
            print(row_str)
    
    # Freeze section  
    print("\\midrule")
    print("\\multicolumn{7}{l}{\\textbf{Freeze}} \\\\")
    freeze_data = combined_results[
        (combined_results['Pretext Task'] == ssl_tech) & 
        (combined_results['FT Strategy'] == 'Freeze')
    ]
    
    for dataset in datasets:
        dataset_freeze = freeze_data[freeze_data['Dataset'] == dataset]
        if not dataset_freeze.empty:
            row_str = f"{dataset:<10}"
            for encoder in encoders:
                encoder_data = dataset_freeze[dataset_freeze['Backbone Clean'] == encoder]
                if not encoder_data.empty:
                    row = encoder_data.iloc[0]
                    mean_pct = row['Mean'] * 100
                    std_pct = row['Std'] * 100
                    net_score = row['Net Score']
                    row_str += f"& {mean_pct:.1f}±{std_pct:.1f} ({net_score:+d}) "
                else:
                    row_str += "& - "
            row_str += "\\\\"
            print(row_str)
    
    print("\\bottomrule")
    print("\\end{tabular}")
    print(f"\\caption{{Performance for {ssl_tech} across datasets and encoders. Mean±Std accuracy (\\%) and net score in parentheses.}}")
    print(f"\\label{{tab:{ssl_tech.lower()}_detailed}}")
    print("\\end{table}")
    print()

# Debug table to check data availability
print("\n" + "="*80)
print("=== Data Availability Check ===")
print()

availability_data = []
for ssl_tech in pretext_tasks:
    for ft_strategy in ['Full Finetune', 'Freeze']:
        for encoder in encoders:
            for dataset in datasets:
                data_exists = not combined_results[
                    (combined_results['Pretext Task'] == ssl_tech) & 
                    (combined_results['FT Strategy'] == ft_strategy) &
                    (combined_results['Backbone Clean'] == encoder) &
                    (combined_results['Dataset'] == dataset)
                ].empty
                if data_exists:
                    availability_data.append({
                        'SSL': ssl_tech,
                        'Strategy': ft_strategy,
                        'Encoder': encoder,
                        'Dataset': dataset
                    })

availability_df = pd.DataFrame(availability_data)
print("Data points found:")
print(f"Total: {len(availability_df)}")
print("\nBreakdown by encoder:")
print(availability_df['Encoder'].value_counts())
print("\nBreakdown by SSL technique:")
print(availability_df['SSL'].value_counts())
print("\nBreakdown by strategy:")
print(availability_df['Strategy'].value_counts())